### 🧠 What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### ✅ Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
#from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableMap, RunnablePassthrough


c:\Users\SamarjitMahi\OneDrive - ZapCom Solutions Pvt. ltd\Pictures\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Step 1: Load and embed the document
loader = TextLoader("langchain_crewai_dataset.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4, "lambda_mult": 0.7})

C:\Users\SamarjitMahi\AppData\Local\Temp\ipykernel_21092\2324071921.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8120.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001D60281C550>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D60281CF50>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [8]:
# Step 3: Query decomposition
decomposition_prompt = PromptTemplate.from_template("""
You are an AI assistant. Decompose the following complex question into 2 to 4 smaller sub-questions for better document retrieval.

Question: "{question}"

Sub-questions:
""")
decomposition_chain = decomposition_prompt | llm | StrOutputParser()

In [9]:
query = "How does LangChain use memory and agents compared to CrewAI?"
decomposition_question=decomposition_chain.invoke({"question": query})


In [10]:
print(decomposition_question)

To better understand the differences between LangChain and CrewAI, I'll break down the complex question into smaller sub-questions:

1. **What is the memory architecture used in LangChain?**
   - This sub-question helps to identify the specific memory management capabilities of LangChain, allowing for a comparison with CrewAI.

2. **What kind of agents are supported in LangChain?**
   - This sub-question focuses on the types of agents used in LangChain, enabling a comparison with CrewAI's supported agents.

3. **How does CrewAI manage memory and agents?**
   - This sub-question provides a baseline for comparison by understanding how CrewAI handles memory and agents, which can be contrasted with LangChain's approach.

These sub-questions can help to gather specific information about LangChain and CrewAI, facilitating a more accurate comparison between the two.


In [14]:
# Step 4: QA chain per sub-question using modern LCEL
qa_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.

Context:
{context}

Question: {input}
""")

qa_chain = (
    qa_prompt
    | llm
    | StrOutputParser()
)

# Step 5: Full RAG pipeline with query decomposition
def full_query_decomposition_rag_pipeline(user_query):
    # Decompose the query into sub-questions
    sub_qs_text = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-•1234567890. ").strip() for q in sub_qs_text.split("\n") if q.strip()]

    results = []
    for subq in sub_questions:
        docs = retriever.invoke(subq)
        context = "\n\n".join(doc.page_content for doc in docs)
        answer = qa_chain.invoke({"input": subq, "context": context})
        results.append(f"Q: {subq}\nA: {answer}")

    return "\n\n".join(results)



In [13]:
# Step 6: Run
query = "How does LangChain use memory and agents compared to CrewAI?"
final_answer = full_query_decomposition_rag_pipeline(query)
print("✅ Final Answer:\n")
print(final_answer)

✅ Final Answer:

Q: Based on the original question, I can break it down into the following sub-questions for better document retrieval:
A: Based on the provided context, it seems you're looking to improve document retrieval using LangChain. Here are some potential sub-questions to help you break down the original question:

1. How can I integrate LangChain with vector databases like FAISS, Chroma, Pinecone, and Weaviate for semantic search within large document corpora?
2. In Retrieval-Augmented Generation (RAG), how can I fetch and inject external knowledge into the LLM (Large Language Model) using LangChain?
3. What are the benefits of hybrid retrieval in LangChain, and how can I combine keyword-based (sparse) retrieval methods like BM25 with embedding-based (dense) retrieval?
4. How can I leverage LangChain's support for hybrid retrieval to improve recall in my document retrieval system, especially in catching both exact term matches and semantically similar content?
5. Are there an